# Benchmark de encoders - CODEFEST Ad Astra 2026

Compara tres modelos *encoder-only* multilingües para elegir cuál usar en la Fase 4.

| Modelo | dim | arquitectura | prefijos |
|---|---|---|---|
| `ibm-granite/granite-embedding-311m-multilingual-r2` | 768 | ModernBERT | no |
| `ibm-granite/granite-embedding-97m-multilingual-r2` | 384 | ModernBERT | no |
| `intfloat/multilingual-e5-base` | 768 | XLM-RoBERTa | sí (`query:` / `passage:`) |

Los tres son *encoder-only*, según su `config.json`, como exige la Sección 8.3 del reto.

**Bloques:**

1. **Ficha técnica** - dimensión, límite de tokens, arquitectura y tamaño.
2. **Idiomas cruzados** - capacidad para relacionar consultas en español con documentos en inglés.
3. **Prefijos de e5** - efecto de usar o no los prefijos.
4. **Tiempo en CPU** - tiempo estimado para indexar el corpus completo.

**Antes de correr:** todo se ejecuta en CPU. El bloque 1 descarga los tres modelos (~2.5 GB en total). La primera descarga puede tardar varios minutos. Después se usa la copia local.

## 0. Preparación

### Instalar dependencias

In [ ]:
# !pip install sentence-transformers torch pandas

### Cómo se descargan los modelos

HuggingFace guarda cada modelo como una carpeta con pesos, tokenizer y archivos de configuración. La librería los descarga la primera vez y los guarda en una caché local:

```text
~/.cache/huggingface/hub/          (Linux y macOS)
C:\Users\<usuario>\.cache\huggingface\hub\   (Windows)
```

Las siguientes ejecuciones reutilizan esa copia.

La celda siguiente descarga los modelos de forma explícita para mostrar el progreso y el tamaño de cada uno. `ignore_patterns` evita descargar las versiones ONNX y OpenVINO, que no se usan.

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

MODELOS = {
    "granite-311m": {
        "repo": "ibm-granite/granite-embedding-311m-multilingual-r2",
        "prefijo_consulta": "",
        "prefijo_texto": "",
    },
    "granite-97m": {
        "repo": "ibm-granite/granite-embedding-97m-multilingual-r2",
        "prefijo_consulta": "",
        "prefijo_texto": "",
    },
    "e5-base": {
        "repo": "intfloat/multilingual-e5-base",
        "prefijo_consulta": "query: ",
        "prefijo_texto": "passage: ",
    },
}

SOBRAN = ["onnx/*", "openvino/*", "*.onnx", "*.h5", "*.msgpack", "*.ot", "*.tflite"]

# Los modelos granite son ModernBERT, soportado desde transformers 4.48.
# Con una versión anterior fallan al cargar con un error poco claro.
import transformers
print("transformers", transformers.__version__, "(los granite necesitan >= 4.48)\n")

def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6

for alias, cfg in MODELOS.items():
    print(f"--- {alias} ---")
    ruta = snapshot_download(cfg["repo"], ignore_patterns=SOBRAN)
    cfg["ruta_local"] = ruta
    print(f"    {tamano_mb(ruta):,.0f} MB en {ruta}\n")

### Cargar el mini-corpus

Se cargan doce fragmentos reales del corpus y tres consultas del reto (q006, q026, q042), con relevancia de 0 a 3. Los fragmentos que no aparecen en `relevance` tienen relevancia 0.

Las tres consultas están en español, pero los fragmentos correctos de q006 y q026 están en inglés. Es el escenario que se quiere evaluar en el bloque 2.

In [ ]:
import json
from pathlib import Path

def raiz_repo(inicio=None):
    '''Sube por el árbol de carpetas hasta encontrar la raíz del repo.'''
    actual = Path(inicio or Path.cwd()).resolve()
    for candidata in [actual, *actual.parents]:
        if (candidata / "tests" / "fixtures" / "mini_corpus.json").exists():
            return candidata
    raise FileNotFoundError(
        "No encuentro tests/fixtures/mini_corpus.json. "
        "Abre el notebook desde dentro del repo, o fija RAIZ a mano."
    )

RAIZ = raiz_repo()
CORPUS = json.loads((RAIZ / "tests" / "fixtures" / "mini_corpus.json").read_text(encoding="utf-8"))

CONSULTAS = CORPUS["consultas"]
TEXTOS = CORPUS["texts"]

print(f"raíz del repo: {RAIZ}")
print(f"{len(CONSULTAS)} consultas, {len(TEXTOS)} fragmentos")
print()
for c in CONSULTAS:
    oro = [t["id"] for t in TEXTOS if t["relevance"].get(c["id"], 0) >= 2]
    idiomas = {t["language"] for t in TEXTOS if t["id"] in oro}
    print(f"{c['id']}  correctos: {len(oro)} ({', '.join(sorted(idiomas))})  |  {c['text'][:70]}...")
print()
print("fragmentos por idioma:", {i: sum(1 for t in TEXTOS if t["language"] == i) for i in {t["language"] for t in TEXTOS}})

## Bloque 1 - Ficha técnica

Se consulta cada modelo para obtener cuatro datos: dimensión, límite de tokens, arquitectura y número de parámetros.

**`max_seq_length`** es el límite configurado que se aplica realmente. Puede ser menor que el límite de la arquitectura. Si un texto supera `max_seq_length`, se trunca sin generar un error. Por eso se comparan ambos valores.

**La dimensión** indica el tamaño del vector. Afecta el tamaño del índice y la velocidad de búsqueda.

**La arquitectura** confirma que el modelo es *encoder-only*. Es la evidencia que se incluye en el informe.

**Los parámetros** sirven para estimar el tiempo de inferencia en CPU.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer

modelos_cargados = {}
fichas = []

for alias, cfg in MODELOS.items():
    m = SentenceTransformer(cfg["repo"], device="cpu")
    modelos_cargados[alias] = m

    conf = m[0].auto_model.config
    fichas.append({
        "modelo": alias,
        "arquitectura": conf.architectures[0] if conf.architectures else "?",
        "dim": m.get_sentence_embedding_dimension(),
        "max_seq_length": m.max_seq_length,
        "techo_arquitectura": getattr(conf, "max_position_embeddings", None),
        "params_M": round(sum(p.numel() for p in m.parameters()) / 1e6),
        "prefijos": "sí" if cfg["prefijo_consulta"] else "no",
    })

ficha = pd.DataFrame(fichas).set_index("modelo")
ficha

**Cómo leer esta tabla.**

Compara `max_seq_length` con `techo_arquitectura`. Si `max_seq_length` es 512 aunque la arquitectura soporte mucho más, el modelo trunca a 512. Si el límite configurado es mayor, puede trabajar con chunks más largos que e5.

En `arquitectura` se esperan modelos basados en BERT, ModernBERT o XLM-RoBERTa. Una arquitectura con `ForCausalLM` corresponde a un decoder y queda descartada.

`max_seq_length` se mide en **tokens**, no en palabras. En español, 512 tokens suelen equivaler a unas 200–300 palabras, según el texto.

In [ ]:
# Cuántos tokens gasta realmente cada tokenizer, por idioma.
# Sirve para traducir el límite de tokens a un límite de palabras usable en la Fase 3.
filas_tok = []
for alias, m in modelos_cargados.items():
    for idioma in sorted({t["language"] for t in TEXTOS}):
        muestras = [t["text"] for t in TEXTOS if t["language"] == idioma]
        razones = [
            len(m.tokenizer.encode(x, add_special_tokens=False)) / len(x.split())
            for x in muestras
        ]
        filas_tok.append({
            "modelo": alias,
            "idioma": idioma,
            "tokens_por_palabra_medio": round(sum(razones) / len(razones), 2),
            "tokens_por_palabra_max": round(max(razones), 2),
            "palabras_que_caben": int(m.max_seq_length / max(razones)),
        })

pd.DataFrame(filas_tok).set_index(["modelo", "idioma"])

`palabras_que_caben` indica cuántas palabras pueden entrar sin truncamiento y se usa en la Fase 3. El cálculo toma el caso más costoso observado. Si el resultado supera con margen las 250 palabras, el límite del reto es el que manda.

## Utilidades de medición

Tres funciones usadas en los bloques 2 y 3.

`puntuar` convierte consultas y fragmentos en vectores y calcula todas las similitudes. Con `normalize_embeddings=True`, el producto punto equivale a la similitud coseno, igual que en FAISS con `IndexFlatIP`.

`ranking` ordena los fragmentos de una consulta y marca los correctos.

`resumir` calcula cuatro métricas por modelo. Se explican en el bloque 2.

In [ ]:
import numpy as np
import pandas as pd

def puntuar(modelo, cfg, consultas, textos):
    '''Matriz (consultas x textos) de similitud coseno.'''
    vq = modelo.encode(
        [cfg["prefijo_consulta"] + c["text"] for c in consultas],
        normalize_embeddings=True, batch_size=8, show_progress_bar=False,
    )
    vt = modelo.encode(
        [cfg["prefijo_texto"] + t["text"] for t in textos],
        normalize_embeddings=True, batch_size=8, show_progress_bar=False,
    )
    return np.asarray(vq) @ np.asarray(vt).T


def ranking(matriz, consultas, textos, id_consulta):
    i = next(k for k, c in enumerate(consultas) if c["id"] == id_consulta)
    filas = [
        {
            "puesto": 0,
            "fragmento": t["id"],
            "idioma": t["language"],
            "relevancia": t["relevance"].get(id_consulta, 0),
            "puntaje": round(float(matriz[i, j]), 4),
        }
        for j, t in enumerate(textos)
    ]
    filas.sort(key=lambda f: -f["puntaje"])
    for puesto, f in enumerate(filas, start=1):
        f["puesto"] = puesto
    return pd.DataFrame(filas).set_index("puesto")


def resumir(matriz, consultas, textos, etiqueta):
    reciprocos, aciertos1, margenes, sesgos = [], [], [], []

    for i, c in enumerate(consultas):
        puntajes = matriz[i]
        rel = np.array([t["relevance"].get(c["id"], 0) for t in textos])
        idiomas = np.array([t["language"] for t in textos])

        orden = np.argsort(-puntajes)
        posiciones_oro = [p for p, j in enumerate(orden, start=1) if rel[j] >= 2]
        reciprocos.append(1 / posiciones_oro[0] if posiciones_oro else 0.0)
        aciertos1.append(1 if rel[orden[0]] >= 2 else 0)

        oro, ruido = puntajes[rel >= 2], puntajes[rel < 2]
        margenes.append(float(oro.min() - ruido.max()) if len(oro) and len(ruido) else np.nan)

        # Sesgo de idioma: solo entre fragmentos NO relevantes, para que la
        # diferencia no se explique por el contenido.
        ruido_es = puntajes[(rel < 2) & (idiomas == "es")]
        ruido_en = puntajes[(rel < 2) & (idiomas == "en")]
        sesgos.append(float(ruido_es.mean() - ruido_en.mean()) if len(ruido_es) and len(ruido_en) else np.nan)

    return {
        "config": etiqueta,
        "MRR": round(float(np.mean(reciprocos)), 3),
        "acierto@1": f"{sum(aciertos1)}/{len(consultas)}",
        "margen": round(float(np.nanmean(margenes)), 4),
        "sesgo_idioma": round(float(np.nanmean(sesgos)), 4),
    }

## Bloque 2 - Idiomas cruzados

Las 50 consultas del reto están en español y los corpus de los fenómenos 1 y 2 están en inglés. Un sesgo hacia el idioma de la consulta puede subir documentos en español que no son relevantes y bajar documentos en inglés que sí lo son. Esto afecta el F1@3.

Las tres consultas se comparan con los doce fragmentos. Los fragmentos de las otras consultas funcionan como distractores y permiten detectar este sesgo.

In [ ]:
matrices = {}
resultados_b2 = []

for alias, m in modelos_cargados.items():
    matrices[alias] = puntuar(m, MODELOS[alias], CONSULTAS, TEXTOS)
    resultados_b2.append(resumir(matrices[alias], CONSULTAS, TEXTOS, alias))

pd.DataFrame(resultados_b2).set_index("config")

**Cómo leer cada columna.**

**`MRR`** - posición del primer fragmento correcto. Vale 1.0 si aparece primero, 0.5 si aparece segundo y 0.33 si aparece tercero.

**`acierto@1`** - proporción de consultas cuyo primer resultado es correcto.

**`margen`** - distancia entre el peor fragmento correcto y el mejor incorrecto. Un valor **positivo** indica separación entre ambos grupos; uno **negativo** indica mezcla. Este valor ayuda a elegir el umbral θ de similitud en la Fase 6.

**`sesgo_idioma`** - diferencia de puntuación entre distractores en español y en inglés. Solo se consideran fragmentos no relevantes. Un valor cercano a 0 es lo ideal. Un valor **positivo** indica una ventaja para los textos en español.

A continuación se muestra el detalle de cada consulta.

In [ ]:
for id_consulta in [c["id"] for c in CONSULTAS]:
    texto = next(c["text"] for c in CONSULTAS if c["id"] == id_consulta)
    print("=" * 100)
    print(f"{id_consulta}: {texto[:95]}")
    print("=" * 100)
    for alias in modelos_cargados:
        print(f"\n-- {alias} --")
        print(ranking(matrices[alias], CONSULTAS, TEXTOS, id_consulta).head(5).to_string())
    print()

In [ ]:
filas_disp = []
for alias, M in matrices.items():
    plano = M.flatten()
    filas_disp.append({
        "modelo": alias,
        "puntaje_min": round(float(plano.min()), 3),
        "puntaje_max": round(float(plano.max()), 3),
        "desviacion": round(float(plano.std()), 4),
    })

disp = pd.DataFrame(filas_disp).set_index("modelo")
disp["margen"] = pd.DataFrame(resultados_b2).set_index("config")["margen"]
disp["margen_normalizado"] = (disp["margen"] / disp["desviacion"]).round(2)
disp

En estas tablas se buscan tres señales:

1. Los fragmentos con relevancia 3 deberían aparecer arriba.
2. Los fragmentos de **otra** consulta pueden indicar confusión temática.
3. Puntajes muy cercanos indican poca separación. Por ejemplo, 0.87 frente a 0.85 entre el primero y el quinto resultado.

## Bloque 3 - Prefijos de e5

`multilingual-e5-base` se entrenó con `query:` delante de la consulta y `passage:` delante del documento. Los prefijos forman parte de la entrada esperada por el modelo.

Se prueban tres configuraciones: prefijos correctos, sin prefijos y prefijos intercambiados. Así se mide el efecto de omitirlos y se obtiene un dato para el informe técnico.

In [ ]:
variantes = {
    "e5 con prefijos":      {"prefijo_consulta": "query: ",   "prefijo_texto": "passage: "},
    "e5 sin prefijos":      {"prefijo_consulta": "",          "prefijo_texto": ""},
    "e5 prefijos al revés": {"prefijo_consulta": "passage: ", "prefijo_texto": "query: "},
}

resultados_b3 = [
    resumir(puntuar(modelos_cargados["e5-base"], cfg, CONSULTAS, TEXTOS), CONSULTAS, TEXTOS, nombre)
    for nombre, cfg in variantes.items()
]

pd.DataFrame(resultados_b3).set_index("config")

Si las tres filas son casi idénticas, la prueba puede no tener suficiente resolución con tres consultas y doce fragmentos. En ese caso, se mantienen los prefijos recomendados por el modelo y se indica en el informe que el efecto no pudo medirse a esta escala.

## Bloque 4 - Tiempo en CPU

En CPU, el tiempo de inferencia puede ser un criterio importante. El proceso se repite al cambiar el chunking, corregir extractores o ajustar parámetros.

Se mide el costo por fragmento y se extrapola al corpus completo. Se debe ajustar `CHUNKS_ESTIMADOS` cuando se conozca el tamaño real del corpus. Por ejemplo, 400 documentos con 60 chunks cada uno producen 24.000 chunks.

In [ ]:
import time

CHUNKS_ESTIMADOS = 30_000
REPETICIONES = 3

filas_tiempo = []
for alias, m in modelos_cargados.items():
    cfg = MODELOS[alias]
    entradas = [cfg["prefijo_texto"] + t["text"] for t in TEXTOS]

    m.encode(entradas[:2], normalize_embeddings=True, show_progress_bar=False)  # calentar

    inicio = time.perf_counter()
    for _ in range(REPETICIONES):
        m.encode(entradas, normalize_embeddings=True, batch_size=8, show_progress_bar=False)
    transcurrido = time.perf_counter() - inicio

    por_fragmento = transcurrido / (REPETICIONES * len(entradas))
    filas_tiempo.append({
        "modelo": alias,
        "ms_por_fragmento": round(por_fragmento * 1000, 1),
        "fragmentos_por_seg": round(1 / por_fragmento, 1),
        "horas_corpus_completo": round(por_fragmento * CHUNKS_ESTIMADOS / 3600, 2),
    })

pd.DataFrame(filas_tiempo).set_index("modelo")

Dos advertencias sobre esta extrapolación.

Los fragmentos del mini-corpus rondan las 200 palabras. Si la Fase 3 usa chunks de 140 palabras, el costo real puede ser menor.

El tiempo tampoco depende solo del número de parámetros. También influyen los núcleos disponibles y la carga del equipo. Conviene ejecutar esta celda con el equipo en reposo.

## Resumen para decidir

In [ ]:
resumen = (
    pd.DataFrame(resultados_b2).set_index("config")
      .join(pd.DataFrame(filas_tiempo).set_index("modelo")[["ms_por_fragmento", "horas_corpus_completo"]])
      .join(ficha[["dim", "max_seq_length", "params_M"]])
)
resumen

### Cómo decidir con esta tabla

**1. Sesgo de idioma.** Un `sesgo_idioma` alto es una señal para descartar el modelo. Dos de los tres fenómenos están en inglés y todas las consultas están en español.

**2. Margen.** El margen muestra mejor que el MRR si los puntajes separan los fragmentos relevantes de los demás. También ayuda a fijar el umbral θ de la Fase 6.

**3. Tiempo.** Si una corrida completa supera unas dos horas, se debe valorar si la mejora en calidad compensa el menor ritmo de iteración.

**4. Dimensión.** 384 frente a 768 implica un índice más pequeño y búsquedas más rápidas. Si granite-97m ofrece una calidad similar, su menor tamaño es una ventaja.

### Antes de cerrar la decisión

Este notebook mide **tres consultas y doce fragmentos**. Sirve para descartar modelos débiles, pero no para elegir con confianza entre modelos parejos.

Si dos modelos quedan igualados:

- El reto permite fusionar varios encoders con RRF o CombSUM. Si los modelos cometen errores distintos, la fusión puede superar a cada uno por separado.
- La decisión final debe hacerse con el harness de la Fase 7, usando las 30 consultas propias con relevancia graduada.

El informe debe incluir el criterio de descarte y no solo el modelo seleccionado.

In [ ]:
DESTINO = RAIZ / "entrega"
DESTINO.mkdir(exist_ok=True)

resumen_final = (
    pd.DataFrame(resultados_b2).set_index("config")
      .join(pd.DataFrame(filas_tiempo).set_index("modelo"))
      .join(disp[["puntaje_min", "puntaje_max", "desviacion", "margen_normalizado"]])
      .join(ficha)
)
resumen_final.index.name = "modelo"
resumen_final.to_csv(DESTINO / "benchmark_encoders_resumen.csv", encoding="utf-8")

pd.DataFrame(resultados_b3).set_index("config").to_csv(
    DESTINO / "benchmark_encoders_prefijos_e5.csv", encoding="utf-8")

rankings = pd.concat(
    [ranking(matrices[a], CONSULTAS, TEXTOS, c["id"]).assign(modelo=a, consulta=c["id"])
     for a in matrices for c in CONSULTAS]
).reset_index()[["modelo", "consulta", "puesto", "fragmento", "idioma", "relevancia", "puntaje"]]
rankings.to_csv(DESTINO / "benchmark_encoders_rankings.csv", index=False, encoding="utf-8")

for f in sorted(DESTINO.glob("benchmark_encoders_*.csv")):
    print(f"{f.name:45s} {f.stat().st_size:>7,} bytes")